In [0]:
# 1. Standard Imports
from pyspark.sql.functions import avg, count, col, round, desc

# 2. Load the Enriched Silver Data
df_enriched = spark.table("railway_analytics.silver.fact_train_delays_enriched")

# 3. Calculate Train-Level Performance
# We group by train_no and trainName to get one row per train
df_top_delayed_trains = (df_enriched
    .groupBy("train_no", "station_name_raw")
    .agg(
        count("*").alias("total_stops_recorded"),
        round(avg("delay"), 2).alias("average_delay_minutes")
    )
    
    .filter(col("total_stops_recorded") > 100)
    .orderBy(desc("average_delay_minutes"))
)

# 4. Save the Final Gold Asset
(df_top_delayed_trains.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("railway_analytics.gold.top_delayed_trains"))

# 5. Display Results
display(spark.table("railway_analytics.gold.top_delayed_trains").limit(10))

train_no,station_name_raw,total_stops_recorded,average_delay_minutes
1028,PRAYAGRAJ CHHEOKI - PCOI,402,1516.5
18125,PURI - PURI,730,1470.84
14816,SHRI GANGANAGAR - SGNR,730,1463.22
7030,AMBASA - ABSA,104,869.76
7030,DHARMANAGAR - DMR,104,869.06
7030,AGARTALA - AGTL,104,861.54
7030,NEW KARIMGANJ - NKMG,104,860.78
7030,NEW HAFLONG - NHLG,104,850.06
7030,BADARPUR JN - BPB,104,838.64
1666,NARMADAPURAM - NDPM,104,834.78
